In [2]:
import findspark
findspark.init()
findspark.find()
import os

os.environ['HADOOP_CONF_DIR'] = '/etc/hadoop/conf'
os.environ['YARN_CONF_DIR'] = '/etc/hadoop/conf'
os.environ['JAVA_HOME']='/usr'
os.environ['SPARK_HOME'] ='/usr/lib/spark'
os.environ['PYTHONPATH'] ='/usr/local/lib/python3.8'

import pyspark
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from datetime import datetime, timedelta

from pyspark.sql.functions import count, row_number, desc
from pyspark.sql.window import Window
import sys

from pyspark.sql.functions import explode, col, countDistinct
from pyspark.sql.functions import lit

In [4]:
spark = SparkSession.builder \
    .appName("Geo") \
    .master("local") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.cores", "1") \
    .config("spark.executor.instances", "5") \
    .getOrCreate()

In [12]:
# Загрузка событий и geo.csv
events = spark.read.parquet("/user/master/data/geo/events")
geo = spark.read.csv(
    "/user/marbiana/data/geo.csv",
    header=True,
    inferSchema=True,
    sep=";"  
)

In [14]:
# Перевод координат в радианы
from pyspark.sql.functions import radians, col

# Преобразуем координаты сообщений
events = events.withColumn("lat_rad", radians(col("lat"))) \
               .withColumn("lon_rad", radians(col("lon")))

# Преобразуем координаты городов
geo = geo.withColumn("city_lat_rad", radians(col("lat"))) \
         .withColumn("city_lon_rad", radians(col("lng")))

In [15]:
# Картезий произведение и расстояние по формуле гаверсинуса
from pyspark.sql.functions import sin, cos, pow, sqrt, asin, lit

joined = events.join(geo)

distance_expr = 2 * 6371 * asin(
    sqrt(
        pow(sin((joined.lat_rad - joined.city_lat_rad)/2), 2) +
        cos(joined.lat_rad) * cos(joined.city_lat_rad) *
        pow(sin((joined.lon_rad - joined.city_lon_rad)/2), 2)
    )
)

joined = joined.withColumn("distance", distance_expr)

In [11]:
# Выбор ближайшего города
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("event.message_id").orderBy("distance")
events_with_city = joined.withColumn("rn", row_number().over(window)) \
                         .filter("rn = 1") \
                         .drop("rn")

In [19]:
# Найти act_city — последний город
from pyspark.sql.functions import col, max as spark_max


latest_events = events_with_city.groupBy("event.message_from").agg(
    spark_max("event.datetime").alias("max_time")
).select(
    col("message_from").alias("user_id"),  
    col("max_time")
)


events_with_city = events_with_city.select(
    col("event.message_from").alias("user_id"),
    col("event.datetime"),
    col("city")
)


act_city = latest_events.join(
    events_with_city,
    on=[
        latest_events.user_id == events_with_city.user_id,
        latest_events.max_time == events_with_city.datetime
    ],
    how="inner"
).select(
    events_with_city.user_id,
    events_with_city.city.alias("act_city")
)

In [21]:
# Найти home_city — самый долгий город более 27 дней
from pyspark.sql.functions import to_date, countDistinct, row_number, desc
from pyspark.sql.window import Window

# Тут названия ровно такие, какие ты реально спроецировал!
visits = events_with_city.select(
    col("user_id"),
    col("city"),
    to_date(col("datetime")).alias("date")
).distinct()

visits_per_city = visits.groupBy("user_id", "city").agg(
    countDistinct("date").alias("days")
)

home_city = visits_per_city.filter("days >= 27") \
    .withColumn(
        "rank",
        row_number().over(Window.partitionBy("user_id").orderBy(desc("days")))
    ) \
    .filter("rank = 1") \
    .select(
        col("user_id"),
        col("city").alias("home_city")
    )

In [22]:
# Подсчитать travel_count и travel_array
from pyspark.sql.functions import collect_list

travel = visits.orderBy("date").groupBy("user_id").agg(
    count("*").alias("travel_count"),
    collect_list("city").alias("travel_array")
)


In [24]:
# Посчитать local_time
from pyspark.sql.functions import from_utc_timestamp, max as spark_max

local_time = (
    events_with_city
    .groupBy("user_id")
    .agg(spark_max("datetime").alias("max_time"))
    .join(
        events_with_city,
        (col("datetime") == col("max_time")) & (col("user_id") == col("user_id")),
        how="inner"
    )
    .select(
        col("user_id"),
        from_utc_timestamp(col("datetime"), "Australia/Sydney").alias("local_time")
    )
)

2025-07-09 18:43:59,419 WARN sql.Column: Constructing trivially true equals predicate, ''user_id = 'user_id'. Perhaps you need to use aliases.


AnalysisException: Reference 'user_id' is ambiguous, could be: user_id, user_id.;

In [ ]:
# витрина в разрезе пользователей
final = act_city \
    .join(home_city, "user_id", "left") \
    .join(travel, "user_id", "left") \
    .join(local_time, "user_id", "left")